# Collector Missing Data Spots (Polars)

Lists collector polling gaps and estimates missed polls and rows using the Polars analysis path.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

missing_data = load_polars_script("polars_collector_missing_data_spots", "collector-missing-data-spots.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
SOURCE = None
LIMIT = 20
MIN_OBSERVATIONS = 1
GAP_MULTIPLIER = 2.0
MIN_MISSING_MINUTES = 0.0


In [ ]:
settings = missing_data.ReportSettings(
    db=DB,
    limit=LIMIT,
    min_observations=MIN_OBSERVATIONS,
).resolved()
polls = missing_data.load_collector_polls(settings, source=SOURCE)
spots = missing_data.build_missing_spots(
    polls,
    gap_multiplier=GAP_MULTIPLIER,
    min_missing_minutes=MIN_MISSING_MINUTES,
)
summary = missing_data.summarize_missing_spots(spots, polls)
summary


In [ ]:
spots.head(LIMIT)


In [ ]:
if summary.is_empty():
    print("No collector polls found.")
else:
    fig = px.bar(
        summary.sort("total_missing_min"),
        x="total_missing_min",
        y="source",
        orientation="h",
        title="Estimated missing-data duration by collector source",
        labels={"source": "Source", "total_missing_min": "Estimated missing minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
